# GitHub Issues Gateway Assignment Notebook

## CRUD on Issues + Webhook Handling + OpenAPI Contract + Tests

**Author:** Arti  
**Stack:** Python, FastAPI, GitHub REST API, SQLite, Pytest, OpenAPI 3.1, Docker

This notebook documents the implementation, validation, and evidence for the assignment.

## 1. Project Structure

```text
github-issues-service/
├── app/
│   ├── __init__.py
│   ├── main.py
│   ├── github_client.py
│   ├── webhook.py
│   ├── storage.py
│   └── models.py
├── tests/
│   ├── test_api.py
│   ├── test_webhook.py
│   └── test_storage.py
├── openapi.yaml
├── requirements.txt
├── Dockerfile
├── .env.example
├── .gitignore
├── README.md
├── DESIGN.md
├── pytest.ini
└── Makefile
```

**Evidence:** Add the `tree /F` screenshot to the final report.

## 2. Security and Environment Variables

Configuration is loaded from environment variables:

```text
GITHUB_TOKEN=<local secret>
GITHUB_OWNER=<GitHub username>
GITHUB_REPO=github-issues-service
WEBHOOK_SECRET=<local webhook secret>
PORT=8000
```

Security controls:
- No hard-coded GitHub token
- `.env` excluded via `.gitignore`
- HMAC SHA-256 webhook verification
- Constant-time signature comparison
- Secrets/signatures should never be logged

**Evidence:** Include `.gitignore` screenshot, but never the actual `.env` contents.

## 3. API Endpoints

| Method | Endpoint | Purpose |
|---|---|---|
| GET | `/healthz` | Health check |
| POST | `/issues` | Create issue |
| GET | `/issues` | List issues |
| GET | `/issues/{number}` | Get issue |
| PATCH | `/issues/{number}` | Update / close / reopen |
| POST | `/issues/{number}/comments` | Add comment |
| POST | `/webhook` | Receive GitHub webhook |
| GET | `/events` | View persisted webhook events |

GitHub does not provide issue deletion, so **closing an issue is the delete-equivalent**.

## 4. Health Check

```http
GET /healthz
```

Expected:

```json
{"status":"ok"}
```

The service also adds an `X-Request-ID` response header.

**Verified:** HTTP 200 returned successfully.

## 5. Create Issue — Real GitHub Integration

Request:

```json
{
  "title": "Integration Test Issue",
  "body": "Created through the GitHub Issues Gateway API for assignment testing.",
  "labels": []
}
```

Observed:

```json
{
  "number": 1,
  "state": "open",
  "title": "Integration Test Issue",
  "body": "Created through the GitHub Issues Gateway API for assignment testing.",
  "labels": []
}
```

**Status:** 201 Created

**Evidence:** Insert Swagger POST `/issues` screenshot.

## 6. Read and List Issues

Single issue:

```http
GET /issues/1
```

List with pagination:

```http
GET /issues?state=all&page=1&per_page=10
```

Supported query options:
- `state=open|closed|all`
- `labels`
- `page`
- `per_page` up to 100

The GitHub `Link` header is preserved when present.

## 7. Update, Close, and Reopen

Update:

```json
{
  "title": "Updated Integration Test Issue",
  "body": "Updated through the GitHub Issues Gateway API during integration testing.",
  "state": "open"
}
```

Close:

```json
{"state":"closed"}
```

Reopen:

```json
{"state":"open"}
```

All operations use:

```http
PATCH /issues/1
```

## 8. Add Comment

```http
POST /issues/1/comments
```

Request:

```json
{
  "body": "Integration test comment added through the GitHub Issues Gateway API."
}
```

**Status:** 201 Created

The updated issue and comment were also verified in the GitHub UI.

## 9. Webhook Verification

Webhook requests use:

```text
X-Hub-Signature-256
X-GitHub-Event
X-GitHub-Delivery
```

The service verifies:

```text
HMAC-SHA256(WEBHOOK_SECRET, raw_request_body)
```

Verified behavior:
- Valid signature → **204 No Content**
- Invalid signature → **401 Unauthorized**
- `issues`, `issue_comment`, and `ping` supported
- Unknown event/action rejected

## 10. Webhook Persistence and Idempotency

A valid event was persisted and visible through:

```http
GET /events
```

Observed event:

```text
id           : assignment-delivery-001
event        : issues
action       : opened
issue_number : 1
```

Re-sending the same delivery ID did not create a duplicate record, demonstrating retry-safe behavior.

## 11. OpenAPI 3.1 Contract

Validation command:

```powershell
python -c "import yaml; data=yaml.safe_load(open('openapi.yaml')); print('OpenAPI version:', data.get('openapi')); print('Title:', data.get('info',{}).get('title')); print('Paths:', len(data.get('paths',{})))"
```

Observed:

```text
OpenAPI version: 3.1.0
Title: GitHub Issues Gateway
Paths: 6
```

Swagger UI is available at:

```text
http://127.0.0.1:8000/docs
```

## 12. Automated Tests

Run:

```powershell
pytest --cov=app --cov-report=term-missing -v
```

Coverage result:

```text
app/__init__.py          100%
app/github_client.py      58%
app/main.py               91%
app/models.py            100%
app/storage.py           100%
app/webhook.py            92%
TOTAL                     86%
```

The assignment target was **≥80%**, and the implementation achieved **86%**.

## 13. Error Mapping

| Condition | Service Response |
|---|---|
| GitHub 401 | 401 authentication failure |
| GitHub 403 | 403 access error |
| Rate limit / Retry-After | 429 |
| GitHub 404 | 404 |
| Other 4xx | 400 validation error |
| GitHub 5xx | 503 service unavailable |

## 14. Observability

Implemented:
- `X-Request-ID`
- webhook delivery IDs
- `/healthz`
- `/events`
- upstream error mapping
- no secret/signature logging

## 15. Docker

Build:

```powershell
docker build -t github-issues-service .
```

Run:

```powershell
docker run --rm --name github-issues-service-container -p 8000:8000 --env-file .env github-issues-service
```

Verify:

```powershell
Invoke-RestMethod http://127.0.0.1:8000/healthz
```

**Docker execution evidence is pending local Docker engine setup.**

## 16. Architecture

```text
Client / Swagger
      |
      v
FastAPI REST Layer
      |
      +-------------------+
      |                   |
      v                   v
GitHubClient          Webhook Handler
      |                   |
      v                   v
GitHub REST API       HMAC Verification
                          |
                          v
                     SQLite Event Store
```

## 17. Integration Test Summary

| Test | Result |
|---|---|
| Health check | PASS |
| Create issue | PASS |
| Get issue | PASS |
| List issues | PASS |
| Update title/body | PASS |
| Close issue | PASS |
| Reopen issue | PASS |
| Create comment | PASS |
| GitHub UI verification | PASS |
| Valid webhook signature | PASS |
| Invalid webhook signature | PASS |
| Webhook persistence | PASS |
| Duplicate delivery handling | PASS |
| OpenAPI 3.1 validation | PASS |
| Unit coverage ≥80% | PASS — 86% |
| Docker execution | Pending |

## 18. Screenshot Checklist for Final Report

Include:
1. GitHub repository page
2. `.gitignore` showing `.env`
3. Swagger UI
4. `/healthz`
5. POST `/issues` — 201
6. GET `/issues/1`
7. GET `/issues`
8. PATCH update
9. Close
10. Reopen
11. POST comment — 201
12. GitHub UI issue/comment
13. Valid webhook — 204
14. `/events`
15. Invalid signature — 401
16. Idempotency verification
17. OpenAPI 3.1 validation
18. Final tests + 86% coverage
19. `tree /F`
20. Docker build
21. Docker container startup
22. Container `/healthz`

## 19. Conclusion

The GitHub Issues Gateway implements the required CRUD-style issue lifecycle, comments, webhook verification, event persistence, idempotency, OpenAPI 3.1 documentation, error mapping, health checks, request IDs, and automated testing.

The project achieved **86% test coverage**, exceeding the required 80% target. Real GitHub integration was validated for create, read, list, update, close, reopen, and comment operations.